In [3]:
import numpy as np
import pandas as pd
from scipy import stats
import random
from datetime import datetime, timedelta
from faker import Faker

# ── Reproductibilité ─────────────────────────────────────────────────────────
random.seed(42)
np.random.seed(42)
fake = Faker()
Faker.seed(42)

# ── Paramètres ──────────────────────────────────────────────────────────────
NB_USERS = 500
INTERESTS = ["fitness", "musique", "technologie", "cuisine", "voyage", "cinema"]
INTEREST_PROBS = [0.25, 0.22, 0.18, 0.15, 0.12, 0.08]  # distribution inégale

ACTIONS = {
    "fitness":     ["watched workout video", "bought protein powder",
                    "liked gym post", "joined gym"],
    "musique":     ["watched music video", "bought headphones",
                    "liked song", "attended concert", "streamed album"],
    "technologie": ["watched AI talk", "read tech blog", "bought laptop",
                    "bought smartwatch", "liked AI article",
                    "contributed to open source"],
    "cuisine":     ["watched cooking show", "bought ingredients",
                    "liked recipe post", "tried new recipe",
                    "bought kitchen gadget"],
    "voyage":      ["watched travel vlog", "booked flight", "bought luggage",
                    "liked travel photo", "checked into hotel"],
    "cinema":      ["watched movie", "watched trailer", "bought streaming sub",
                    "rated film", "bought cinema ticket"],
}

# ── Génération des profils ───────────────────────────────────────────────────
def generate_users(n=NB_USERS):
    records = []
    for i in range(n):
        age = int(np.random.normal(loc=35, scale=10))
        nb_interests = random.randint(1, 3)
        interests = list(np.random.choice(
            INTERESTS, size=nb_interests, replace=False, p=INTEREST_PROBS
        ))
        records.append({
            "user_id":    i + 1,
            "name":       fake.name(),
            "age":        age,
            "age_group":  "18-25" if age < 26 else "26-40" if age < 41 else "41+",
            "interests":  interests,
        })
    return pd.DataFrame(records)

# ── Inférence du type d'action ───────────────────────────────────────────────
def infer_action_type(action):
    """Déduit le type d'action à partir du verbe."""
    if isinstance(action, str):
        if action.startswith(("watched", "read", "streamed")):
            return "view"
        if action.startswith("bought") or action == "booked flight":
            return "buy"
        if action.startswith("liked") or action == "rated film":
            return "like"
    return "other"

# ── Génération des journaux d'activité ──────────────────────────────────────
def generate_activity_logs(users_df, logs_per_user=10):
    logs = []
    base_date = datetime(2024, 1, 1)
    for _, user in users_df.iterrows():
        for _ in range(logs_per_user):
            category = random.choice(user["interests"])
            action = random.choice(ACTIONS[category])
            action_type = infer_action_type(action)

            delta = timedelta(days=random.randint(0, 180),
                              hours=random.randint(0, 23))
            ts = base_date + delta
            logs.append({
                "user_id":     user["user_id"],
                "action":      action,
                "action_type": action_type,
                "category":    category,
                "timestamp":   ts,
                "hour":        ts.hour,
            })
    return pd.DataFrame(logs)

# ── Injection de bruit (doublons + NaN) ──────────────────────────────────────
def inject_noise(df, dup_fraction=0.03, nan_fraction=0.02):
    """Injecte des doublons et des NaN pour simuler des données réelles."""
    df_noisy = df.copy()

    # ── Doublons ──
    n_dup = int(len(df_noisy) * dup_fraction)
    dup_rows = df_noisy.sample(n=n_dup, random_state=42)
    df_noisy = pd.concat([df_noisy, dup_rows], ignore_index=True)

    # ── Valeurs manquantes ──
    n_nan = int(len(df_noisy) * nan_fraction)
    nan_indices = df_noisy.sample(n=n_nan, random_state=42).index
    for idx in nan_indices:
        col = random.choice(["action", "category", "hour"])
        df_noisy.loc[idx, col] = np.nan

    return df_noisy

# ── Exécution ────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    users_df = generate_users()
    logs_df  = generate_activity_logs(users_df)
    logs_df  = inject_noise(logs_df)

    print("=== Utilisateurs ===")
    print(users_df.head())
    print(f"\nShape : {users_df.shape}")
    print(f"Valeurs manquantes :\n{users_df.isnull().sum()}\n")

    print("=== Journaux d'activité ===")
    print(logs_df.head())
    print(f"\nShape : {logs_df.shape}")
    print(f"Valeurs manquantes :\n{logs_df.isnull().sum()}")
    print(f"Doublons : {logs_df.duplicated().sum()}")

=== Utilisateurs ===
   user_id             name  age age_group                        interests
0        1     Allison Hill   39     26-40  [cuisine, technologie, fitness]
1        2      Noah Rhodes   33     26-40                        [fitness]
2        3  Angie Henderson   50       41+                        [fitness]
3        4    Daniel Wagner   42       41+        [cinema, voyage, fitness]
4        5  Cristian Santos   30     26-40           [musique, technologie]

Shape : (500, 5)
Valeurs manquantes :
user_id      0
name         0
age          0
age_group    0
interests    0
dtype: int64

=== Journaux d'activité ===
   user_id                 action action_type     category  \
0        1       tried new recipe       other      cuisine   
1        1     bought ingredients         buy      cuisine   
2        1         read tech blog        view  technologie   
3        1  bought protein powder         buy      fitness   
4        1        watched AI talk        view  technologi